## Test the Conversion to Lunar Time in C++

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pylupnt as pnt
import matplotlib.pyplot as plt
from tqdm import tqdm

# TCG -> TCL

In [ ]:
fig_dir = pnt.get_output_dir() / "lunar_time"
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
# tcg_start = pnt.gregorian_to_time(2020, 1, 1, 0, 0, 0.0)
# tcg_end = pnt.gregorian_to_time(2022, 1, 1, 0, 0, 0.0)
# mjd_start = pnt.gregorian_to_mjd(2020, 1, 1, 0, 0, 0.0)
# mjd_end = pnt.gregorian_to_mjd(2022, 1, 1, 0, 0, 0.0)

tcg_start = pnt.gregorian_to_time(2027, 1, 1, 0, 0, 0.0)
tcg_end = pnt.gregorian_to_time(2028, 1, 1, 0, 0, 0.0)
mjd_start = pnt.gregorian_to_mjd(2027, 1, 1, 0, 0, 0.0)
mjd_end = pnt.gregorian_to_mjd(2028, 1, 1, 0, 0, 0.0)

t_dt = 0.01 * 86400.0
n_tcg = int((tcg_end - tcg_start) / t_dt) + 1
tcg_array = np.linspace(tcg_start, tcg_end, n_tcg)
mjd_array = np.linspace(mjd_start, mjd_end, n_tcg)

# tcl_array = np.zeros_like(tcg_array)
# for i, t_tcg in tqdm(enumerate(tcg_array), total=len(tcg_array)):
#     tcl_array[i] = pnt.tcg_to_tcl(t_tcg)
tcl_array = pnt.tcg_to_tcl(tcg_array)
diff = tcl_array - tcg_array

# remove secular drift
# fit ax + b to diff
A = np.vstack([tcg_array, np.ones(len(tcg_array))]).T
m, c = np.linalg.lstsq(A, diff, rcond=None)[0]
diff_periodic = diff - (m * tcg_array + c)
print(f"Secular drift: {m*1e6*86400} micro-sec / sec")
print(f"Offset: {c*1e6} micro-sec")

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(mjd_array, diff_periodic * 1e6)
ax.set_xlabel("Modified Julian Date [days]", fontsize=14)
ax.set_ylabel(r"TCL - TCG [$\mu s$]", fontsize=14)
ax.set_title("TCL - TCG (Secular Drift Removed)", fontsize=14)
ax.grid()
plt.tight_layout()
plt.savefig(fig_dir / "tcl_tcg_difference.pdf")

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
latlon = [(0.0, 0.0), (np.pi / 2, 0.0), (0.0, np.pi / 2)]
t_tai = pnt.convert_time(tcg_array, pnt.TCG, pnt.TAI)
for lat, lon in latlon:
    x_pa = pnt.R_MOON * np.cos(lat) * np.cos(lon)
    y_pa = pnt.R_MOON * np.cos(lat) * np.sin(lon)
    z_pa = pnt.R_MOON * np.sin(lat)
    x_pa = np.array([x_pa, y_pa, z_pa])
    lat_deg = np.degrees(lat)
    lon_deg = np.degrees(lon)
    print(f"Location lat={lat_deg} lon={lon_deg} x_pa={x_pa}")
    xyz_mci = pnt.convert_frame(t_tai, x_pa, pnt.MOON_PA, pnt.MOON_CI)
    print(f"Converted to MCI: {xyz_mci.shape}")
    proper_time_correction = pnt.get_proper_time_correction_tcl(tcg_array, xyz_mci)
    ax.plot(
        mjd_array, proper_time_correction * 1e6, label=f"lat={lat_deg} lon={lon_deg}"
    )
ax.set_xlabel("MJD [days]")
ax.set_ylabel("Proper Time Correction [micro-seconds]")
ax.set_title("Proper Time Correction for TCL at Different Locations on the Moon")
ax.grid()
ax.legend()
plt.tight_layout()
plt.savefig(fig_dir / "proper_time_correction.pdf")
plt.show()

In [ ]:
tcg_array_back = pnt.tcl_to_tcg(tcl_array)
diff_back = tcg_array_back - tcg_array

# plot the back conversion error
plt.figure(figsize=(8, 4))
plt.plot(mjd_array, diff_back * 1e12)
plt.xlabel("MJD [days]")
plt.ylabel("TCG_back - TCG [pico-seconds]")
plt.title("Error in Back Conversion from TCL to TCG")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# manual versus built-in comparison
tai_array = pnt.convert_time(tcg_array, pnt.TCG, pnt.TAI)
tcl_array_builtin = pnt.convert_time(tai_array, pnt.TAI, pnt.TCL)
diff_builtin = tcl_array_builtin - tcl_array
# print(f"Difference between built-in and manual TCL conversion: {diff_builtin*1e6} micro-sec")

plt.figure(figsize=(8, 4))
plt.plot(mjd_array, diff_builtin * 1e12)
plt.xlabel("MJD [days]")
plt.ylabel("TCL_builtin - TCL_manual [pico-seconds]")
plt.title("Difference between Built-in and Manual TCL Conversion")
plt.grid()
plt.tight_layout()
plt.show()